In [15]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
import k3d
import sys
import os

import trimesh
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import PlaceholderNet, GeneralNet
from models.residuals import grad_x_f, hess_x_f
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cpu' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

In [16]:
# mesh = trimesh.load("bun_zipper.ply")
mesh = trimesh.load("rocker-arm.off")
pts = torch.tensor(mesh.vertices, dtype=torch.float64)
center = pts.mean(dim=0)
pts -= center
scale = pts.norm(dim=1).max()
pts /= scale
model = PlaceholderNet(ks=[3, 32, 32, 32, 1], act=torch.sin)
model.load_state_dict(torch.load('3,32,32,32,1,sin,rockerarm,gn', map_location=torch.device('cpu')))
model.double()

PlaceholderNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [17]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=pts.min(dim=0).values-0.01,
    bbox_max=pts.max(dim=0).values+0.15,
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()
model.double()

Output()

PlaceholderNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [19]:
grad_x_f(model, model.params, pts)

tensor([[[-0.0463,  0.1845,  0.4278]],

        [[-0.0684, -0.0071,  0.4281]],

        [[ 0.0440,  0.1738,  0.4997]],

        ...,

        [[ 0.0898, -0.2716, -0.8033]],

        [[-0.0466, -0.2431, -0.7809]],

        [[-0.1822, -0.0552, -0.8210]]], dtype=torch.float64,
       grad_fn=<ViewBackward0>)